**Import các thư viện cần thiết**

In [52]:
import pandas as pd;
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN

**Tìm hiểu các trường dữ liệu**

In [53]:
df = pd.read_csv("./Dataset_v1.9.csv")
df.dropna(inplace=True)
df.head()

,id,Label,mint_count_per_week,burn_count_per_week,mint_ratio,swap_ratio,burn_ratio,mint_mean_period,swap_mean_period,burn_mean_period,swap_in_per_week,swap_out_per_week,swap_rate,lp_avg,lp_std,lp_creator_holding_ratio,lp_lock_ratio,token_burn_ratio,token_creator_holding_ratio,number_of_token_creation_of_creator
0,0x3cd1c0b98be4451ca51265bbaeb76cf7b31e1c02,False,44.071475,16.466820,0.148132,0.796520,0.055348,0.034508,0.074894,0.221980,100.871903,136.103828,0.741002,0.119760,0.126458,0.000000e+00,0.000000,0.000000e+00,1.302617e-01,1
1,0x0c52de7bb69edd144d58d772fab1cd196919f5ef,False,13.813171,6.316279,0.169770,0.752600,0.077630,0.040580,0.050907,0.083953,42.502064,18.732391,2.266527,0.332226,0.431320,0.000000e+00,0.000000,4.000000e-07,1.000000e-08,1
2,0xa00d47b4b304792eb07b09233467b690db847c91,False,27.366327,17.779613,0.068019,0.887790,0.044191,0.131459,0.384409,0.191228,165.965722,191.224332,0.867757,0.377358,4.649864,0.000000e+00,0.064726,0.000000e+00,0.000000e+00,1
3,0x114e40ba90e9d8b002bc6936e3299daa393017bb,False,10.559028,1.912477,0.056257,0.933553,0.010189,0.036312,0.077704,0.140362,90.398212,84.822399,1.065397,0.467290,4.754048,1.480000e-08,0.693921,3.748588e-03,0.000000e+00,1
4,0x40829a59080a12f16bb8fba22354a6a42c810aab,False,9.662390,6.450369,0.024123,0.959774,0.016104,0.088733,0.206316,0.171454,204.911119,179.530891,1.141202,0.476190,5.466877,0.000000e+00,0.790202,0.000000e+00,1.000000e-08,1


**Từ dữ liệu bên trên có thể thấy có các trường:**
- Id: địa chỉ ví / contract
- Label: Được thực hiện trong 1 khoảng thời gian giới hạn, dựa trên uniswapV2
    + True = Scam token tính từ giao dịch đầu đến khi xảy ra rug-pull
    + False = Normal token (hoặc không phải rug-pull) tính từ giao dịch đầu đến lần quét (lấy data) cuối cùng
- min_count_per_week: số lượng token phát hành mới của contract trong tuần
- burn_count_per_week: số lượng token bị đốt mỗi tuần
- mint_ratio: tỷ lệ phát hành
- swap_ratio: tỷ lệ trao đổi giữa token và stablecoin (có thể là usd, ...)
- mint_mean_period: thời gian trung bình phát hành token
- swap_mean_period: thời gian trung bình giữa các lần trao đổi token
- burn_mean_period: thời gian trung bình giữa các lần đốt token
- swap_in_per_week: số lượng token trao đổi sang stablecoin / tuần
- swap_out_per_week: số lượng token được stablecoin đổi thành / tuần
- swap_rate: Tỷ giá trao đổi giữa token và stablecoin đó
- lp_avg: giá trị thanh khoản trung bình
- lp_std: độ lệch chuẩn của thanh khoản
- lp_creator_holding_ratio: tỷ lệ thanh khoản của người tạo ra đang nắm giữ
- lp_lock_ratio: tỷ lệ thanh khoản được khỏa trong smart contract
- token_burn_ratio: tỷ lệ token đang bị đốt
- token_creator_holding_ratio: tỷ lệ token mà người tạo ra đang nắm giữ
- number_of_token_creation_of_creator: tổng số token được tạo bởi người tạo token

**Chuyển Label của dữ liệu từ true - false sang 0 - 1**

In [54]:
label_encoder = LabelEncoder()
df["Label"] = label_encoder.fit_transform(df["Label"])
df.head()

,id,Label,mint_count_per_week,burn_count_per_week,mint_ratio,swap_ratio,burn_ratio,mint_mean_period,swap_mean_period,burn_mean_period,swap_in_per_week,swap_out_per_week,swap_rate,lp_avg,lp_std,lp_creator_holding_ratio,lp_lock_ratio,token_burn_ratio,token_creator_holding_ratio,number_of_token_creation_of_creator
0,0x3cd1c0b98be4451ca51265bbaeb76cf7b31e1c02,0,44.071475,16.466820,0.148132,0.796520,0.055348,0.034508,0.074894,0.221980,100.871903,136.103828,0.741002,0.119760,0.126458,0.000000e+00,0.000000,0.000000e+00,1.302617e-01,1
1,0x0c52de7bb69edd144d58d772fab1cd196919f5ef,0,13.813171,6.316279,0.169770,0.752600,0.077630,0.040580,0.050907,0.083953,42.502064,18.732391,2.266527,0.332226,0.431320,0.000000e+00,0.000000,4.000000e-07,1.000000e-08,1
2,0xa00d47b4b304792eb07b09233467b690db847c91,0,27.366327,17.779613,0.068019,0.887790,0.044191,0.131459,0.384409,0.191228,165.965722,191.224332,0.867757,0.377358,4.649864,0.000000e+00,0.064726,0.000000e+00,0.000000e+00,1
3,0x114e40ba90e9d8b002bc6936e3299daa393017bb,0,10.559028,1.912477,0.056257,0.933553,0.010189,0.036312,0.077704,0.140362,90.398212,84.822399,1.065397,0.467290,4.754048,1.480000e-08,0.693921,3.748588e-03,0.000000e+00,1
4,0x40829a59080a12f16bb8fba22354a6a42c810aab,0,9.662390,6.450369,0.024123,0.959774,0.016104,0.088733,0.206316,0.171454,204.911119,179.530891,1.141202,0.476190,5.466877,0.000000e+00,0.790202,0.000000e+00,1.000000e-08,1


**Kiểm tra dataset có bao nhiêu dữ liệu được gán nhãn rug-pull và non rug-pull**

In [55]:
df['Label'].value_counts()

1    16462
0     1834
Name: Label, dtype: int64

**Loại bỏ cột id và label để tạo ra bộ dữ liệu huấn luyện và test**

In [56]:
features = df.drop(['Label', 'id'], axis=1)
target = df['Label']
(x_train, x_test, y_train, y_test) = train_test_split(features, target, test_size=0.2, random_state=42)

**Sử dụng mô hình KNN để huấn luyện mô hình và độ chính xác của mô hình**

In [57]:
kneighbors = KNeighborsClassifier()

# Huấn luyện mô hình
kneighbors.fit(x_train, y_train)
y_predict = kneighbors.predict(x_test) 
print(accuracy_score(y_true=y_test, y_pred=y_predict))

0.9456284153005464


**Sử dụng mô hinh random forest để huấn luyện và độ chính xác của mô hình**

In [59]:

random_forest = RandomForestClassifier()
random_forest.fit(x_train, y_train)
y_predict_random_forest = random_forest.predict(x_test)
print(accuracy_score(y_true=y_test, y_pred=y_predict_random_forest))

0.98224043715847


**Sử dụng mô hình phân cụm để phân thành 2 cụm và kiểm tra các cụm có bao nhiêu thành phần**

In [71]:

kmeans = KMeans(n_clusters=2) # Do cần tìm 2 cụm rugpull và non rugpull
kmeans_clusters = kmeans.fit(features)
pd.Series(kmeans_clusters.labels_).value_counts()


0    18282
1       14
dtype: int64

**Sử dụng DBSCAN để phân cụm dữ liệu và kiểm tra có bao nhiêu cụm được tạo thành**

In [72]:
dbscan = DBSCAN()
dbscan_cluster = dbscan.fit(features)
pd.Series(dbscan_cluster.labels_).value_counts()

-1      11972
 46       218
 32       193
 38       151
 5        129
        ...  
 130        5
 274        5
 236        5
 262        5
 219        4
Length: 279, dtype: int64

**Sử dụng phân phối gaussian để phân cụm và tổng số các thành phần mỗi cụm**

In [75]:
from sklearn.mixture import GaussianMixture

# Tạo mô hình GMM với số lượng cụm là 2 (có thể điều chỉnh theo yêu cầu của bạn)
gmm_model = GaussianMixture(n_components=2)
gmm_model.fit(features)
gmm_clusters = gmm_model.predict(features)
pd.Series(gmm_clusters).value_counts()


0    13561
1     4735
dtype: int64

In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 1.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 7.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.9/773.9 kB 6.0 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.0/169.0 kB 5.6 MB/s eta 0:00:00


In [2]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 2.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 4.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 5.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 6.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 4.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 5.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 4.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━

In [7]:
from transformers import BertForSequenceClassification
import torch

bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncaseed', num_labels=2,
    output_attentions=False,
    output_hidden_states=False)
optimizer = torch.optim.Adam(bert_model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

ImportError: 
BertForSequenceClassification requires the PyTorch library but it was not found in your environment.
However, we were able to find a TensorFlow installation. TensorFlow classes begin
with "TF", but are otherwise identically named to our PyTorch classes. This
means that the TF equivalent of the class you tried to import would be "TFBertForSequenceClassification".
If you want to use TensorFlow, please use TF classes instead!

If you really do want to use PyTorch please go to
https://pytorch.org/get-started/locally/ and follow the instructions that
match your environment.


In [ ]:
bert_model.train()
